# Linear Regression

# Using Tensors and Autograd

In [64]:
import torch

importamos el dataset

In [65]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42) #por defecto tiene 0.25 de test_size
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

convertimos a tensores y normalizamos

In [66]:
X_train=torch.FloatTensor(X_train)
X_valid=torch.FloatTensor(X_valid)
X_test=torch.FloatTensor(X_test)
means=X_train.mean(dim=0, keepdims=True)
stds=X_train.std(dim=0, keepdims=True)
X_train=(X_train-means)/stds
X_valid=(X_valid-means)/stds
X_test=(X_test-means)/stds

convertimos las targets a tensores, y le hacemos reshape para que se vuelca un vector columna

In [67]:
y_train=torch.FloatTensor(y_train).reshape(-1,1) #el -1 es para que pytorch calcule automaticamente el numero de filas
y_valid=torch.FloatTensor(y_valid).reshape(-1,1)
y_test=torch.FloatTensor(y_test).reshape(-1,1)

In [68]:
y_train

tensor([[1.4420],
        [1.6870],
        [1.6210],
        ...,
        [0.6800],
        [0.6130],
        [1.9700]])

creamos los parametros de nuestro modelo de regresion lineal

In [69]:
torch.manual_seed(42)
n_features=X_train.shape[1] #hay 8 features de entrada
w=torch.randn((n_features,1),requires_grad=True) #weight
b=torch.tensor(0.,requires_grad=True) #bias

## entrenamos el moldelo (usamos bacth gradient descent(BGD))
- primero definimos el learning rate
- despues corremos 20 epocas
- despues calculamos la prediccion (y_pred) y el mean square error para el loss
- entonces corremos loss.backward()  para calcular los gradientes de el loss con respecto a cada parametro del modelo
- usamos los gradientes b.grad y w.grad para hacer el gradiente descent step dentro de torch.no_grad()
- una vez que hicimos el gradient descent step reseteamos los gradientes a zero(importante)
- y por ultimo imprimimos el numero de la epoca y el loss de cada epoca. el metodo item() extrae el valor de un tesnor escalar

In [70]:
learning_rate=0.4
n_epochs=20
for epoch in range(n_epochs):
  y_pred=X_train @ w + b
  loss=((y_pred-y_train)**2).mean()
  loss.backward()
  with torch.no_grad():
    b-=learning_rate * b.grad
    w-=learning_rate * w.grad
    b.grad.zero_()
    w.grad.zero_()
  print(f"epoch {epoch+1}/{n_epochs}, loss: {loss.item()}")

epoch 1/20, loss: 16.158456802368164
epoch 2/20, loss: 4.8793745040893555
epoch 3/20, loss: 2.255225419998169
epoch 4/20, loss: 1.3307634592056274
epoch 5/20, loss: 0.9680691957473755
epoch 6/20, loss: 0.8142675757408142
epoch 7/20, loss: 0.7417045831680298
epoch 8/20, loss: 0.7020701169967651
epoch 9/20, loss: 0.6765918731689453
epoch 10/20, loss: 0.6577965021133423
epoch 11/20, loss: 0.6426151990890503
epoch 12/20, loss: 0.6297222971916199
epoch 13/20, loss: 0.6184942126274109
epoch 14/20, loss: 0.6085968613624573
epoch 15/20, loss: 0.5998216867446899
epoch 16/20, loss: 0.592018723487854
epoch 17/20, loss: 0.5850691795349121
epoch 18/20, loss: 0.578873336315155
epoch 19/20, loss: 0.573345422744751
epoch 20/20, loss: 0.5684100389480591


hagamos prediccion de las tres primeras instancias  del test set

In [71]:
X_new=X_test[:3] #pretendemos que son nuevas instancias
with torch.no_grad():
  y_pred=X_new @ w + b #usamos los parametros entrenados para hacer predicciones
y_pred

tensor([[0.8916],
        [1.6480],
        [2.6577]])

# Using Pytorch's High-Level API

pytorch tiene una implementacion de la regresion lineal en la clase torch.nn.linear

In [72]:
import torch.nn as nn #por convencion este modulo usualmente se importa de esta manera

torch.manual_seed(42)
model=nn.Linear(in_features=n_features,out_features=1)

In [73]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [74]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

ahora que el modelo esta creado, necesitamos crear un optimizador que actualice los parametros del modelo, y debemos escoger una funcion de loss

In [75]:
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
mse=nn.MSELoss()

funcion para entrenar a nuestro modelo
- en pytorch loss function es normalmente referida como criterion para distingirlos de el valor de loss
- el optimizer.step() corresponde a las dos lineas que actualizaban b y w
- optimizer.zero_grad() corresponde a las dos lineas que hacian cero b.grad y w.grad. Aca no necesitamos usar torch.no_grad(), porque eso lo hace automaticamente el optimizer

In [76]:
def train_bgd(model,optimizer,criterion,X_train,y_train,n_epochs):
  for epoch in range(n_epochs):
    y_pred=model(X_train)
    loss=criterion(y_pred,y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"epoch {epoch+1}/{n_epochs}, loss: {loss.item()}")

llamamos a la funcion para entrenar el modelo

In [77]:
train_bgd(model,optimizer,mse,X_train,y_train,n_epochs)

epoch 1/20, loss: 4.3378496170043945
epoch 2/20, loss: 0.7802939414978027
epoch 3/20, loss: 0.6253842115402222
epoch 4/20, loss: 0.6060433983802795
epoch 5/20, loss: 0.5956299304962158
epoch 6/20, loss: 0.587356686592102
epoch 7/20, loss: 0.5802990794181824
epoch 8/20, loss: 0.5741382837295532
epoch 9/20, loss: 0.5687101483345032
epoch 10/20, loss: 0.5639079809188843
epoch 11/20, loss: 0.5596511363983154
epoch 12/20, loss: 0.5558737516403198
epoch 13/20, loss: 0.5525194406509399
epoch 14/20, loss: 0.5495392084121704
epoch 15/20, loss: 0.5468900203704834
epoch 16/20, loss: 0.544533908367157
epoch 17/20, loss: 0.5424376726150513
epoch 18/20, loss: 0.5405716300010681
epoch 19/20, loss: 0.5389097332954407
epoch 20/20, loss: 0.5374288558959961


probamos el modelo

In [78]:
X_new=X_test[:3]
with torch.no_grad():
  y_pred=model(X_new)
y_pred

tensor([[0.8061],
        [1.7116],
        [2.6973]])

# Implementing a Regression MLP
### capas
1. la primera capa debe tener el mismo numero de inputs de nuestros datos(n_features). Sin embargo puede tener cualquiero numero de ouputs
2. tenemos una funcion ReLU que implementa la activacion ReLU a la primera capa
3. la segunda capa oculta debe tener el mismo numero de inputs que los outputs de la capa anterior. Sin embargo puede tener cualquiere cantidad de outputs
4. Otra vez nn.ReLU para implementar la funcion de activacion a la segunda capa oculta
5. finalmente la capa de salida debe tener los mismos inputs que ouputs de la capa anterior. Pero esta vez el numero de ouputs tiene que se igual que la dimension de targets

In [79]:

torch.manual_seed(42)
#capas de nuestro modelo
model=nn.Sequential(
    nn.Linear(n_features,50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)

entrenemos el modelo como lo hicimos antes

In [80]:
leanrning_rate=0.1
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
mse=nn.MSELoss()
train_bgd(model,optimizer,mse,X_train,y_train,n_epochs)

epoch 1/20, loss: 5.045480251312256
epoch 2/20, loss: 13.989859580993652
epoch 3/20, loss: 12.169429779052734
epoch 4/20, loss: 1.6985068321228027
epoch 5/20, loss: 1.3095498085021973
epoch 6/20, loss: 1.273197889328003
epoch 7/20, loss: 1.2230875492095947
epoch 8/20, loss: 1.1453018188476562
epoch 9/20, loss: 1.0272427797317505
epoch 10/20, loss: 0.8872972726821899
epoch 11/20, loss: 0.7877435684204102
epoch 12/20, loss: 0.741396427154541
epoch 13/20, loss: 0.7111529111862183
epoch 14/20, loss: 0.6847622990608215
epoch 15/20, loss: 0.6619062423706055
epoch 16/20, loss: 0.6469722986221313
epoch 17/20, loss: 0.6583961248397827
epoch 18/20, loss: 0.8009870648384094
epoch 19/20, loss: 1.1085376739501953
epoch 20/20, loss: 2.25761342048645


# Implementing Mini-Batch Gradient Descent Using DataLoaders

In [81]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset=TensorDataset(X_train,y_train)
batch_size=32
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

usamos la aceleracion de hardware

In [82]:
if torch.cuda.is_available():
    device="cuda"
    print("Using GPU")
elif torch.backends.mps.is_available():
    device="mps"
    print("Using MPS")
else:
    device="cpu"
    print("Using CPU")

Using GPU


In [92]:
torch.manual_seed(42)
model=nn.Sequential(
    nn.Linear(n_features, 50), nn.ReLU(),
    nn.Linear(50, 40), nn.ReLU(),
    nn.Linear(40, 1)
)

model=model.to(device)


creamos funcion de entrenamineto para implementar mini-bacth GD

In [93]:
def train(model,optimizer,criterion,train_loader,n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss=0.
    for X_batch,y_batch in train_loader:
      X_batch, y_batch=X_batch.to(device), y_batch.to(device)
      y_pred=model(X_batch)
      loss=criterion(y_pred,y_batch)
      total_loss+=loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss=total_loss/len(train_loader)
    print(f"epoch {epoch+1}/{n_epochs}, loss: {mean_loss:.4f}")

entrenamos el modelo

In [ ]:
train(model,optimizer,mse,train_loader,n_epochs)

epoch 1/20, loss: 5.0460
